In [ ]:
import os
import subprocess
from pathlib import Path

import anndata as ad
import colorcet as cc
import pandas as pd
import scanpy as sc
import seaborn as sns
import squidpy as sq
from scipy.sparse import load_npz
from spatialleiden import search_resolution, spatialleiden

from batch_integration.multisample_utils import (
    preprocess_genes,
    preprocess_scaling,
)
from utils import DATA_DIR, RESULT_DIR, SHELL, activate_env_cmd, remove_tick_and_label

In [ ]:
# Settings
seed = 42

In [ ]:
mosta_dir = DATA_DIR / "MOSTA_temporal"
mosta_results = RESULT_DIR / "MOSTA_temporal"

script_dir = Path("batch_integration")

h5ad = mosta_dir / "Mouse_embryo_all_stage.h5ad"
processed_path = mosta_results / "MOSTA_temporal.h5ad"

In [ ]:
mosta_results.mkdir(exist_ok=True, parents=True)

# Prepare dataset

In [ ]:
adata = ad.read_h5ad(h5ad)

adata.obs = adata.obs[["annotation", "timepoint"]]
adata.var = adata.var[[]]
adata.X = adata.layers["count"]
adata.uns["batch_keys"] = ["timepoint"]

del adata.layers

In [ ]:
sc.pp.filter_cells(adata, min_genes=100)
sc.pp.filter_cells(adata, min_counts=250)

preprocess_genes(adata, min_cells=200, batch_key=adata.uns["batch_keys"][0])
preprocess_scaling(adata, seed=seed)

sq.gr.spatial_neighbors(
    adata, library_key=adata.uns["batch_keys"][0], coord_type="grid", n_neighs=4
)

adata

In [ ]:
adata.write_h5ad(processed_path)

In [ ]:
del adata

# Batch integration

In [ ]:
env = os.environ.copy()
env["MPLBACKEND"] = "pdf"  # avoid matplotlib backend issues

In [ ]:
methods = [
    "no_batch_integration",
    "harmony",
    "bbknn",
    "scvi",
]

In [ ]:
for method in methods:
    conda_cmd = activate_env_cmd(method)

    script_path = script_dir / f"{method}.py"
    out_file = mosta_results / f"{method}.npz"

    cmd = f"{script_path} {processed_path} {out_file}"

    # because bbknn uses n neighbors per batch we derease the number of neighbors
    # so its roughly equal to the rest
    if method == "bbknn":
        n = 3  # how many?
        cmd += f" --n_neighs {n}"

    subprocess.run(f"{conda_cmd} && {cmd}", shell=True, executable=SHELL, env=env)

# Clustering

In [ ]:
import time

In [ ]:
adata = ad.read_h5ad(processed_path)

In [ ]:
for file in mosta_results.glob("*.npz"):
    method = file.stem

    print(f"Processing {method}")

    adata.obsp["connectivities"] = load_npz(file)

    start = time.time()
    spatialleiden(
        adata,
        resolution=(0.75, 1),
        directed=(False, False),
        layer_ratio=1.5,  # TODO: change for bbknn?
        n_iterations=3,
    )
    end = time.time()
    print(f"Time: {(end - start) / 60:.2f} min")

    columns = ["spatialleiden"] + list(adata.uns["batch_keys"])

    labels = adata.obs[columns]
    labels.loc[:, ["x", "y"]] = adata.obsm["spatial"]
    labels.to_parquet(mosta_results / f"{method}.parquet")

In [95]:
n_clusters = 45  # number of annotations

for file in mosta_results.glob("*.npz"):
    method = file.stem

    print(f"Processing {method}")

    adata.obsp["connectivities"] = load_npz(file)

    start = time.time()
    # run with few iterations to search the space and then increase for the last run
    spatial_kwargs = dict(
        directed=(False, False), layer_ratio=1.5, n_iterations=2, seed=seed
    )
    resolutions = search_resolution(
        adata,
        n_clusters,
        start=(0.4, 0.4),
        latent_kwargs=dict(
            directed=False,
            obsp="connectivities",
            random_state=seed,  # n_iterations=2
        ),
        spatial_kwargs=spatial_kwargs,
    )
    spatial_kwargs["n_iterations"] = 5
    spatialleiden(adata, resolution=resolutions, **spatial_kwargs)
    end = time.time()
    print(f"Time: {(end - start) / 60:.2f} min")

    columns = ["spatialleiden"] + list(adata.uns["batch_keys"])

    labels = adata.obs[columns]
    labels.loc[:, ["x", "y"]] = adata.obsm["spatial"]
    labels.to_parquet(mosta_results / f"{method}.parquet")

Processing harmony


/home/jovyan/conda_envs/spatialleiden/lib/python3.12/site-packages/spatialleiden/_resolution_search.py:25: UserWarning: Correct resolution not found. Consider increasing the number of iterations or adjusting the step size.
  warn(


TypeError: spatialleiden._resolution_search.search_resolution_spatial() got multiple values for keyword argument 'n_iterations'

# Plotting

In [ ]:
scatter_kwargs = {
    "s": 1,
    "linewidth": 0,
}

In [ ]:
def plot_sections(
    label_file, batch_key: str = "timepoint", anno_key: str = "spatialleiden"
):
    labels = pd.read_parquet(label_file).astype({"spatialleiden": "category"})
    return section_grid(labels, anno_key=anno_key, batch_key=batch_key)


def section_grid(df, anno_key: str, batch_key: str):
    unique_label = df[anno_key].cat.categories
    palette = dict(
        zip(unique_label, sns.color_palette(cc.glasbey, n_colors=len(unique_label)))
    )
    g = sns.FacetGrid(
        df,
        col=batch_key,
        col_wrap=4,
        legend_out=False,
        sharex=False,
        sharey=False,
    )
    _ = g.map_dataframe(
        sns.scatterplot,
        x="x",
        y="y",
        hue=anno_key,
        palette=palette,
        legend=False,
        **scatter_kwargs,
    )

    for ax in g.axes.flat:
        ax.set(aspect=1)
        remove_tick_and_label(ax)

    _ = g.tight_layout()

    return g

## Annotation

In [ ]:
labels = adata.obs[["annotation", "timepoint"]].copy()
labels[["x", "y"]] = adata.obsm["spatial"]
_ = section_grid(labels, "annotation", "timepoint")

## No integration

In [ ]:
fig = plot_sections(mosta_results / "no_batch_integration.parquet")
fig.savefig(mosta_results / "no_batch_integration.png")

## Harmony

In [ ]:
fig = plot_sections(mosta_results / "harmony.parquet")
fig.savefig(mosta_results / "harmony.png")

## scVI

In [ ]:
fig = plot_sections(mosta_results / "scvi.parquet")
fig.savefig(mosta_results / "scvi.png")

## bbknn

In [ ]:
fig = plot_sections(mosta_results / "bbknn.parquet")
fig.savefig(mosta_results / "bbknn.png")